# KV Cache 优化

KV Cache 是 LLM 推理加速的关键技术，通过缓存已计算的 Key 和 Value，避免重复计算。

## 为什么需要 KV Cache？

在自回归生成中：
- 每次只生成 1 个新 token
- 但需要 attend 到所有历史 token
- 如果不缓存，每次都要重新计算所有历史 token 的 K 和 V
- 时间复杂度: O(n²) → O(n)

## 两个阶段

1. **Prefill**: 处理完整的 prompt，初始化 KV Cache
2. **Decode**: 逐个生成 token，增量更新 KV Cache

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from src.kv_cache import KVCache, MultiHeadAttentionWithCache
from src.attention import MultiHeadAttention

torch.manual_seed(42)

## 1. KV Cache 基本使用

In [ ]:
# 创建 KV Cache
batch_size = 2
num_heads = 8
max_seq_len = 100
head_dim = 64

cache = KVCache(batch_size, num_heads, max_seq_len, head_dim)
print("初始状态:")
print(cache)
print(f"缓存长度: {cache.cache_len}")

## 2. Prefill 阶段

In [ ]:
# 模拟 Prefill: 处理 prompt
prompt_len = 10
k_prompt = torch.randn(batch_size, num_heads, prompt_len, head_dim)
v_prompt = torch.randn(batch_size, num_heads, prompt_len, head_dim)

print("Prefill 阶段:")
print(f"输入 K 形状: {k_prompt.shape}")
print(f"输入 V 形状: {v_prompt.shape}")

k_cached, v_cached = cache.update(k_prompt, v_prompt)

print(f"\n更新后:")
print(f"缓存长度: {cache.cache_len}")
print(f"K 缓存形状: {k_cached.shape}")
print(f"V 缓存形状: {v_cached.shape}")

## 3. Decode 阶段

In [ ]:
# 模拟 Decode: 逐个生成 token
print("Decode 阶段:\n")

for i in range(5):
    # 每次只处理 1 个新 token
    k_new = torch.randn(batch_size, num_heads, 1, head_dim)
    v_new = torch.randn(batch_size, num_heads, 1, head_dim)
    
    k_cached, v_cached = cache.update(k_new, v_new)
    
    print(f"Step {i+1}: 缓存长度 = {cache.cache_len}")

## 4. 性能对比：有 vs 无 KV Cache

In [ ]:
def benchmark_without_cache(model, prompt_len, gen_len, d_model):
    """不使用 KV Cache 的推理"""
    tokens = torch.randn(1, prompt_len, d_model)
    
    times = []
    for i in range(gen_len):
        start = time.time()
        with torch.no_grad():
            output, _ = model(tokens, tokens, tokens, use_cache=False)
        times.append((time.time() - start) * 1000)
        
        new_token = torch.randn(1, 1, d_model)
        tokens = torch.cat([tokens, new_token], dim=1)
    
    return times

def benchmark_with_cache(model, prompt_len, gen_len, d_model):
    """使用 KV Cache 的推理"""
    model.reset_cache()
    
    # Prefill
    prompt = torch.randn(1, prompt_len, d_model)
    with torch.no_grad():
        output, _ = model(prompt, prompt, prompt, use_cache=True, start_pos=0)
    
    # Decode
    times = []
    for i in range(gen_len):
        new_token = torch.randn(1, 1, d_model)
        start = time.time()
        with torch.no_grad():
            output, _ = model(new_token, new_token, new_token, use_cache=True)
        times.append((time.time() - start) * 1000)
    
    return times

# 配置
d_model = 512
num_heads = 8
max_seq_len = 2048
prompt_len = 50
gen_len = 20

model = MultiHeadAttentionWithCache(d_model, num_heads, max_seq_len)
model.eval()

print("性能测试中...")
print(f"配置: prompt_len={prompt_len}, gen_len={gen_len}\n")

# 测试
times_no_cache = benchmark_without_cache(model, prompt_len, gen_len, d_model)
times_with_cache = benchmark_with_cache(model, prompt_len, gen_len, d_model)

# 统计
avg_no_cache = np.mean(times_no_cache)
avg_with_cache = np.mean(times_with_cache)
speedup = avg_no_cache / avg_with_cache

print("结果:")
print(f"不使用 KV Cache: {avg_no_cache:.2f} ms/token")
print(f"使用 KV Cache:   {avg_with_cache:.2f} ms/token")
print(f"加速比: {speedup:.2f}x")

## 5. 总结

### KV Cache 的关键点：

1. **原理**: 缓存已计算的 Key 和 Value，避免重复计算
2. **两个阶段**: Prefill (处理 prompt) 和 Decode (生成 token)
3. **性能提升**: 时间复杂度从 O(n²) 降到 O(n)
4. **内存占用**: 与 batch_size、seq_len、num_heads 成正比
5. **权衡**: 用内存换时间

### 完成！

恭喜你完成了 Attention From Scratch 的学习！

你现在应该理解了：
- ✅ Scaled Dot-Product Attention
- ✅ Multi-Head Attention
- ✅ Grouped Query Attention
- ✅ KV Cache 优化

下一步可以：
1. 阅读 TensorRT-LLM XQA 的源码
2. 学习 FlashAttention 的优化技术
3. 实现自己的 Attention 变体